# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-35: MR-NIRP Indoor Exploration

### What this notebook does
Characterises the MR-NIRP Indoor dataset and settles one question: whether it adds anything
the project does not already have. It is the last unexamined dataset in hand, so the question
is asked in full here - what is on the disk, which camera the surviving frames belong to, how
the frames are encoded, which clock dates them, what the pulse oximeter recorded and how
precisely beats can be timed from it, and then what the app's own face, crop, gating and
reading code makes of the frames and how closely its beats follow the reference. The last
section asks why the camera's beats are timed less closely here than in the comparable public
dataset, and tests two candidate explanations against each other.

No frame is displayed. Every result below is a statistic over pixels, and the dataset's terms
are taken to forbid redistribution of its images.

### Dataset
Nowara, Marks, Mansour, Veeraraghavan, *SparsePPG: Towards Driver Monitoring Using
Camera-Based Vital Signs Estimation in Near-Infrared* (CVPR Workshops 2018), distributed by
Rice University as the MR-NIRP Indoor dataset. Eight subjects, each recorded twice indoors:
once sitting still, once moving the head. Two cameras recorded at once, one colour and one
near-infrared at 940 nm, which the folder names carry; a finger pulse oximeter sampled at
60 Hz is the reference. The copy in hand holds 15 recordings - one subject has no moving
recording - and one frame folder per recording, so only one camera's frames survive; which
camera that is, section 1 reads from the data.

Per recording folder `SubjectN_<task>_940/SubjectN_<task>_940/`:
- `RGB/FrameNNNNN.pgm` - one raw frame per file, a binary PGM with a 16-bit container.
- `CameraTimeLogI.txt`, `CameraCyclesLogI.txt`, `CameraFrameI.txt`, `SetupI.log` - per camera
  `I`: the PC's time when each frame arrived, the camera's own clock when it was captured, its
  frame counter, and what the camera reported about its settings.
- `pulseOx.mat` - the oximeter's waveform, the PC's time of each sample, and the sample count.

### Where this sits
The project already has a camera-vital-signs corpus for training and three datasets kept for
evaluation alone. The question for this one is therefore not whether a pulse can be read from
it - that much is expected - but whether it can carry weight no other set carries: longer
still recordings than the two-minute HRV set, a second camera and site for the SDNN question,
or captures the motion gate has to refuse. The final section states what it does and does not
add, and what a store made from it should hold.

## Imports & Setup
The app's own modules are imported from the repository rather than restated, so every face
box, crop, gate and reading below is the one the app would produce from these frames, and
moves if the app moves. The beat alignment comes from the HRV harness for the same reason: an
offset here means what it means there. Thresholds, spans and rules are fixed in this cell,
before anything is measured.

In [1]:
import json
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.optimize import linprog
from scipy.signal import butter, find_peaks, sosfiltfilt, welch

# Walk up from the notebook until the repo root is found, identified by the MediaPipe asset
# (the same test NB_P3_33 uses; cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# The app's code is imported, not restated: every box, crop, gate and reading below is the one
# the app would produce from these frames, and it moves if the app moves. The alignment comes
# from the HRV harness, so a beat offset here means what it means there.
from app.live_vitals.capture.motion import motion_verdict
from app.live_vitals.capture.session import (detection_positions, face_verdict, framing_verdict,
                                             stable_box)
from app.live_vitals.config import (CLIP_LEN, CROP_PADDING, MAX_BOX_SIDE_LOST, MIN_CONFIDENCE,
                                    N_DETECT_FRAMES, SPECTRAL_ROI_FRACTION, TARGET_FPS,
                                    TARGET_SIZE, WINDOW_STRIDE)
from app.live_vitals.estimators.hr_spectral import HRSpectral
from app.live_vitals.preprocess.face_box import make_landmarker
from app.live_vitals.preprocess.frames import crop_resize
from app.live_vitals.scripts.check_hrv_from_video import (MATCH_TOLERANCE_S, align_offset,
                                                          nearest_gaps)
from app.live_vitals.signal.spectral import bandpass, pos, rgb_trace

MRNIRP_ROOT = Path(r'F:\GoogleDriveDatasets\MR_NIPR_Indoor')
RECORDING_DIR = re.compile(r'^Subject(\d+)_(still|motion)_940$')
PGM_HEADER = re.compile(rb'^P5\s+(\d+)\s+(\d+)\s+(\d+)\s', re.S)
# Frame size is read from every file's own header: most recordings are 640x640, and at least
# one subject's are not, which a fixed size would have turned into a decode error or worse.
NOMINAL_FPS = 30.0
CYCLE_WRAP_S = 128.0                         # the camera clock's seconds field wraps every 128 s
SITES = {'00': (0, 0), '01': (0, 1), '10': (1, 0), '11': (1, 1)}   # places in a 2x2 mosaic cell

OX_FS = 60.0                 # the recorder's nominal rate; section 4 measures it
LOSS_LOOKAHEAD = 30          # samples that must all arrive a period late to accept a loss
LOSS_TOL_S = 0.003           # slack on that test, for jitter in the floor itself
CONTROL_LOSS_RATE = 0.01     # injected losses per sample for the positive control
HR_MIN_BPM, HR_MAX_BPM = 35.0, 180.0
BEAT_SPACING = 0.55          # the least a beat may follow the last, as a share of the period
IBI_LOCAL_TOL = 0.25         # an interval this far from the median around it is an artefact
GAP_GUARD = 2                # samples either side of a beat that must be real, not interpolated
HRV_MIN_KEPT = 0.90          # a reference carries HRV with at least this share of its intervals
HRV_MIN_SECONDS = 90.0       # and at least this much of them
FRAME_WORKERS = 8            # frames read, demosaiced and cropped in parallel threads
MAX_RECORDS = None           # set to an int for a trial run

LANDMARKER = make_landmarker()
print('MR-NIRP Indoor root:', MRNIRP_ROOT, '| exists:', MRNIRP_ROOT.exists())
print(f'app contract: crop {TARGET_SIZE}x{TARGET_SIZE} from a box padded by {CROP_PADDING:g}, '
      f'colour trace from its central {SPECTRAL_ROI_FRACTION:.0%}, windows of {CLIP_LEN} frames '
      f'every {WINDOW_STRIDE}, confidence gate {MIN_CONFIDENCE:g}, framing gate '
      f'{MAX_BOX_SIDE_LOST:.0%} of the box side, training rate {TARGET_FPS:g} fps')

MR-NIRP Indoor root: F:\GoogleDriveDatasets\MR_NIPR_Indoor | exists: True
app contract: crop 72x72 from a box padded by 0.6, colour trace from its central 35%, windows of 160 frames every 80, confidence gate 0.65, framing gate 2% of the box side, training rate 30 fps


## 1. Inventory, and which camera the frames belong to
Counts what is on the disk before any of it is trusted. Two cameras were recorded and only
one camera's frames are in this copy, so the first thing to settle is which. The logs answer
it: the colour camera's time log has exactly one line per frame that survives, while the other
camera's lines count frames that are not there. What each camera reported about itself - its
serial, pixel format, frame rate, gain and shutter - is read from its setup log as support.

Two details are read from the data rather than assumed. The frame's size comes from each
file's own PGM header, because the recordings do not all share one resolution and a fixed size
would reshape a frame into a skewed image instead of raising. And the colour camera's log can
run a line or two longer than the frames it stamped, which is what a recorder does when it is
stopped mid-frame, so each recording's log is trimmed to its saved frames before it dates
anything.

In [2]:
SETTINGS = {'serial': r'Serial number (-?\d+)', 'format': r'img format is: (\w+)',
            'fps_set': r'Frame rate is: ([\d.]+) fps', 'gain_db': r'Gain is: ([\d.]+)',
            'shutter_ms': r'Shutter is: ([\d.]+)'}


def frame_shape(path):
    """(width, height, maxval) from a PGM's own header."""
    with open(path, 'rb') as fh:
        match = PGM_HEADER.match(fh.read(64))
    if match is None:
        raise ValueError(f'{path.name} is not a binary PGM')
    width, height, maxval = (int(g) for g in match.groups())
    return width, height, maxval


def count_lines(path):
    """Non-empty lines in a text log, or -1 when the file is missing."""
    if not path.exists():
        return -1
    with open(path, encoding='utf-8', errors='replace') as fh:
        return sum(1 for line in fh if line.strip())


def camera_settings(path):
    """What the camera reported back in its Setup log."""
    text = path.read_text(encoding='utf-8', errors='replace')
    found = {key: re.search(pattern, text) for key, pattern in SETTINGS.items()}
    return {key: (match.group(1) if match else None) for key, match in found.items()}


def scan_recordings(root=MRNIRP_ROOT):
    """One row per recording: its frame count, both cameras' log lengths and their settings.

    Each recording folder holds a second folder of the same name, and the frames and logs sit
    inside that one. Two cameras were recorded, numbered 0 and 1, and only one of them is the
    colour camera whose frames survive - which camera that is, is read from the data below
    rather than assumed.
    """
    rows = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir() and RECORDING_DIR.match(p.name)):
        subject, task = RECORDING_DIR.match(folder.name).groups()
        inner = folder / folder.name
        frames_dir = inner / 'RGB'
        sizes = [e.stat().st_size for e in os.scandir(frames_dir)
                 if e.name.lower().endswith('.pgm')] if frames_dir.is_dir() else []
        width, height, depth = frame_shape(frames_dir / 'Frame00000.pgm') if sizes else (0, 0, 0)
        row = {'subject': f'{int(subject):02d}', 'task': task, 'folder': inner,
               'frames': len(sizes), 'width': width, 'height': height, 'maxval': depth,
               'sizes_equal': len(set(sizes)) <= 1}
        for index in (0, 1):
            row[f'time_lines{index}'] = count_lines(inner / f'CameraTimeLog{index}.txt')
            row[f'cycle_lines{index}'] = count_lines(inner / f'CameraCyclesLog{index}.txt')
            row[f'frame_lines{index}'] = count_lines(inner / f'CameraFrame{index}.txt')
            settings = camera_settings(inner / f'Setup{index}.log')
            row.update({f'{key}{index}': value for key, value in settings.items()})
        rows.append(row)
    return pd.DataFrame(rows)


recordings = scan_recordings()
# The colour camera is the one whose logs have exactly one line per surviving frame.
matches = {index: int((recordings[f'time_lines{index}'] == recordings.frames).sum())
           for index in (0, 1)}
RGB_CAMERA = max(matches, key=matches.get)
recordings['log_lines'] = recordings[f'time_lines{RGB_CAMERA}']

print(f'recordings {len(recordings)} | subjects {recordings.subject.nunique()} | per task '
      f'{recordings.task.value_counts().to_dict()}')
print(f'frames {recordings.frames.sum()} in total, {recordings.frames.min()}-'
      f'{recordings.frames.max()} per recording ({recordings.frames.min() / NOMINAL_FPS:.0f}-'
      f'{recordings.frames.max() / NOMINAL_FPS:.0f} s at a nominal {NOMINAL_FPS:g} fps)')
print(f'frame size: ' + ', '.join(
    f'{w}x{h} at maxval {m} in {n} recordings' for (w, h, m), n in
    recordings.groupby(['width', 'height', 'maxval']).size().items())
    + f' | every frame of a recording the same size: '
      f'{int(recordings.sizes_equal.sum())}/{len(recordings)}')
print(f'\nlogs with one line per surviving frame: camera 0 in {matches[0]}/{len(recordings)} '
      f'recordings, camera 1 in {matches[1]} -> **camera {RGB_CAMERA} is the colour camera**')
for index in (0, 1):
    settings = recordings[[f'serial{index}', f'format{index}', f'fps_set{index}',
                           f'gain_db{index}', f'shutter_ms{index}']].drop_duplicates()
    print(f'  camera {index}: ' + ' | '.join(f'{col.rstrip("01")}={settings.iloc[0][col]}'
                                             for col in settings.columns))
extra = (recordings[f'time_lines{1 - RGB_CAMERA}'] - recordings.frames)
print(f'  the other camera logs {extra.min()}-{extra.max()} lines more than there are colour '
      f'frames: it recorded, but its frames are not in this copy')
print()
recordings['log_extra'] = recordings.log_lines - recordings.frames
print(recordings[['subject', 'task', 'frames', 'width', 'height', 'log_lines', 'log_extra'
                  ]].to_string(index=False))
print(f'\nthe colour camera logs {recordings.log_extra.min()}-{recordings.log_extra.max()} lines '
      f'more than there are saved frames, in {int((recordings.log_extra > 0).sum())} recordings: '
      f'the recorder stamped frames it had not finished writing when it stopped, so each '
      f"recording's log is trimmed to its saved frames before it dates anything.")

recordings 15 | subjects 8 | per task {'still': 8, 'motion': 7}
frames 59269 in total, 1815-7224 per recording (60-241 s at a nominal 30 fps)
frame size: 640x640 at maxval 65535 in 13 recordings, 848x848 at maxval 65535 in 2 recordings | every frame of a recording the same size: 15/15

logs with one line per surviving frame: camera 0 in 1/15 recordings, camera 1 in 10 -> **camera 1 is the colour camera**
  camera 0: serial=-17265397 | format=MONO12 | fps_set=30 | gain_db=0 | shutter_ms=33
  camera 1: serial=-17473548 | format=RAW12 | fps_set=30 | gain_db=0 | shutter_ms=30
  the other camera logs 0-7 lines more than there are colour frames: it recorded, but its frames are not in this copy

subject   task  frames  width  height  log_lines  log_extra
     01 motion    1966    640     640       1966          0
     01  still    5381    640     640       5381          0
     02 motion    1818    640     640       1818          0
     02  still    6305    640     640       6305          0
  

## 2. The raw frame: a 16-bit container, and the mosaic read from the data
Each frame is the sensor's colour mosaic, one value per pixel behind a red, green or blue
filter repeating in a 2x2 cell, written into a 16-bit container whatever the sensor's depth.
Three properties have to be read from the frames themselves.

**Where the value sits in the container.** The 8-bit image is the high byte. Shifting by four
instead leaves 12 bits, and casting that to an 8-bit type wraps around into noise that still
has frame-like statistics - a first probe of this dataset reported a face in none of its
samples for exactly that reason. The largest value any frame reaches says how much of the
container the sensor fills, and therefore its real depth.

**Its byte order.** A 16-bit PGM is big-endian by the format's definition, and the two
libraries disagree about what to do with that: numpy honours an array's byte order, OpenCV
ignores it. A big-endian frame handed straight to the demosaic is read with the halves of
every pixel swapped, and the result still looks like a face - here it raised the mean level
from 21087 to 27953. The frame reader therefore returns a native-order copy, which costs one
copy per frame and removes the trap for every caller.

**Which site holds which colour.** The two green sites sit on one diagonal behind the same
filter, so their levels agree closely while the red and blue sites agree only where the light
happens to balance them; that identifies the layout without displaying anything. OpenCV names
its Bayer codes after a cell two pixels away from the one the data describes, so the code is
chosen by the same synthetic round trip the comparable notebook used, not by its name.

Clipping is read from the raw values, not the 8-bit ones: the high byte maps the top 256 codes
of the container onto 255, so a count of 8-bit pixels at 255 overstates saturation by far.

In [3]:
def read_raw(path):
    """One frame as native-order uint16, with its size and header length read from the file.

    A 16-bit PGM is big-endian by the format's definition, and the copy into the machine's own
    order is not a formality: numpy honours a byte order, OpenCV does not. Handed a big-endian
    array, cv2.cvtColor reads the same bytes as little-endian and demosaics the two halves of
    every pixel swapped, which on these frames raised the mean from 21087 to 27953 and left a
    picture that still looks like a face. The conversion costs one copy per frame and removes
    the trap for every caller. The pixel count is checked against the header, so a short or
    padded file raises here rather than reshaping into a skewed image.
    """
    data = path.read_bytes()
    match = PGM_HEADER.match(data[:64])
    if match is None:
        raise ValueError(f'{path.name} is not a binary PGM')
    width, height, _ = (int(g) for g in match.groups())
    end = match.end()
    if len(data) - end != width * height * 2:
        raise ValueError(f'{path.name} holds {len(data) - end} bytes of pixels, not '
                         f'{width * height * 2}')
    return np.frombuffer(data, dtype='>u2', offset=end,
                         count=width * height).reshape(height, width).astype(np.uint16)


def to_8bit(raw):
    """The high byte of each sample, which is the 8-bit image.

    The container is 16 bits wide whatever the sensor's depth, so the 8-bit image is the top
    byte. Shifting by 4 instead leaves 12 bits, and casting that to uint8 wraps modulo 256 into
    noise that still has frame-like statistics - a first probe of this dataset reported a face
    in none of its samples for exactly that reason. Per-frame rescaling is also avoided: it
    would introduce a brightness modulation of its own, in the band a pulse lives in.
    """
    return (raw >> 8).astype(np.uint8)


def frame_path(folder, index):
    return folder / 'RGB' / f'Frame{index:05d}.pgm'


def mosaic_audit(row, samples=12):
    """Which 2x2 sites hold the greens, and the levels and clipping of one recording.

    The two green sites see nearly the same light, so their levels sit together while the red
    and blue sites do not: that is what identifies the layout without displaying a frame.
    """
    positions = np.linspace(0, row.frames - 1, samples).astype(int)   # spread over the recording
    raws = [read_raw(frame_path(row.folder, int(i))) for i in positions]
    eight = [to_8bit(r) for r in raws]
    levels = {name: float(np.mean([f[r::2, c::2].mean() for f in eight]))
              for name, (r, c) in SITES.items()}
    pairs = {('00', '11'): abs(levels['00'] - levels['11']),
             ('01', '10'): abs(levels['01'] - levels['10'])}
    greens = min(pairs, key=pairs.get)
    return {'subject': row.subject, 'task': row.task, 'size': f'{row.width}x{row.height}',
            'greens': '+'.join(greens),
            'green_gap': pairs[greens], 'other_gap': max(pairs.values()),
            **{f'level_{name}': value for name, value in levels.items()},
            'level_median': float(np.median([np.median(f) for f in eight])),
            'top_of_range': float(np.mean([(r >= 65520).mean() for r in raws])),
            'at_255': float(np.mean([(f >= 255).mean() for f in eight])),
            'raw_max': int(max(int(r.max()) for r in raws))}


mosaic = pd.DataFrame([mosaic_audit(r) for r in recordings.itertuples()])
layout = mosaic.greens.value_counts()
GREENS = tuple(layout.index[0].split('+'))
# OpenCV names a Bayer code after a cell two pixels away from the one the data describes, so
# the code is chosen by the same synthetic round trip NB_P3_33 used, not by its name.
BAYER_CODE = cv2.COLOR_BayerBG2RGB if GREENS == ('01', '10') else cv2.COLOR_BayerGB2RGB
BAYER_NAME = 'COLOR_BayerBG2RGB' if GREENS == ('01', '10') else 'COLOR_BayerGB2RGB'

print(f'the green pair sits at {layout.to_dict()} -> layout '
      f'{"RGGB" if GREENS == ("01", "10") else "GRBG"}, read with cv2.{BAYER_NAME}')
print(f'the green pair differs by {mosaic.green_gap.median():.2f} levels in the median '
      f'recording, the other pair by {mosaic.other_gap.median():.2f}')
print(f'8-bit level, median over sampled frames: {mosaic.level_median.min():.0f}-'
      f'{mosaic.level_median.max():.0f} | samples at the top of the 16-bit range: '
      f'{mosaic.top_of_range.max():.3%} at worst | at 255 after the shift: '
      f'{mosaic.at_255.max():.3%}')
print(f'the largest raw value seen is {mosaic.raw_max.max()} of 65535, which says how much of '
      f'the container the sensor actually fills')
print()
print(mosaic[['subject', 'task', 'size', 'greens', 'level_median', 'level_00', 'level_01',
              'level_10', 'level_11', 'at_255']].round(3).to_string(index=False))

the green pair sits at {'01+10': 15} -> layout RGGB, read with cv2.COLOR_BayerBG2RGB
the green pair differs by 0.05 levels in the median recording, the other pair by 19.09
8-bit level, median over sampled frames: 31-112 | samples at the top of the 16-bit range: 0.556% at worst | at 255 after the shift: 11.413%
the largest raw value seen is 65520 of 65535, which says how much of the container the sensor actually fills

subject   task    size greens  level_median  level_00  level_01  level_10  level_11  at_255
     01 motion 640x640  01+10          42.0    89.205    80.544    80.702    75.395   0.011
     01  still 640x640  01+10          40.0    84.488    75.793    75.883    70.492   0.004
     02 motion 640x640  01+10          52.5    84.747    73.309    73.353    65.655   0.005
     02  still 640x640  01+10          51.0    85.581    73.448    73.369    63.475   0.007
     03 motion 640x640  01+10          48.5    87.058    78.800    78.779    71.577   0.006
     03  still 640x640  01

## 3. Camera timing: which clock dates a frame
Every frame is dated three ways, as in the comparable dataset. One log holds the PC's time
when the frame arrived, one the camera's own clock when it was captured - seconds that wrap
every 128 s - and one the camera's frame counter. The counter shows whether a frame was
dropped, and the camera's clock is immune to whatever the PC was doing; the PC's clock is the
only one the oximeter shares, so it is still needed to bring the two together.

It is not used frame by frame, though. A time written by a busy PC is late by a variable
amount and never early, so the relation between the two clocks is the lower envelope of the
points rather than a least-squares line, which a burst of late stamps drags with it.
`lower_line` fits the line that lies under every point with the least total gap above it, a
small linear programme; what is left above it is how late the PC was for each frame. A frame
is then dated by the camera clock mapped onto the PC's through that line.

Two numbers decide what a store may assume. How far each recording's frames sit from a uniform
grid at its own mean rate says whether frames can be kept as a uniform sequence with one frame
rate, as the app's contract treats them. And how late the PC's stamps run says how much of the
camera-to-reference offset is the recorder rather than the body.

In [4]:
def lower_line(x, y):
    """(a, b) of the line a + b*x lying under every point with the least total gap.

    Times written by a PC are late by a variable amount and never early, so a clock relation is
    the lower envelope of the points. With x centred the total gap depends only on the line's
    height at the centre, so the programme maximises that height.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x0, y0 = x.mean(), y.min()
    xc = x - x0
    sol = linprog(c=[-1.0, 0.0], A_ub=np.column_stack([np.ones_like(xc), xc]), b_ub=y - y0,
                  bounds=[(None, None), (None, None)], method='highs')
    if not sol.success:
        raise RuntimeError(f'lower envelope fit failed: {sol.message}')
    a, b = sol.x
    return float(y0 + a - b * x0), float(b)


def read_camera_logs(folder, index=None):
    """Returns (PC arrival times, camera clock from the first frame, frame counter), in s."""
    index = RGB_CAMERA if index is None else index
    host = np.loadtxt(folder / f'CameraTimeLog{index}.txt', ndmin=1)
    cycles = np.loadtxt(folder / f'CameraCyclesLog{index}.txt', ndmin=1)
    with open(folder / f'CameraFrame{index}.txt', encoding='utf-8', errors='replace') as fh:
        counter = np.array([int(re.search(r'\d+', line).group()) for line in fh if line.strip()])
    step = np.diff(cycles)
    step[step < 0] += CYCLE_WRAP_S           # the seconds field wraps
    return host, np.concatenate([[0.0], np.cumsum(step)]), counter


def frame_clock(folder):
    """Frame times on the PC clock, and the frame rate on the camera's own clock."""
    host, clock, _ = read_camera_logs(folder)
    a, b = lower_line(clock, host - clock)
    return a + (1.0 + b) * clock, (len(clock) - 1) / float(clock[-1])


def clock_for_frames(folder, frames):
    """The frame times of the saved frames, where the logs may run a few lines longer."""
    times, fps = frame_clock(folder)
    return times[:frames], fps


def timing_audit(row):
    """Drops, rate, grid deviation and the PC's lateness for one recording."""
    host, clock, counter = read_camera_logs(row.folder)
    a, b = lower_line(clock, host - clock)
    lag_ms = (host - (a + (1.0 + b) * clock)) * 1000.0
    step_ms = np.diff(clock) * 1000.0
    counter_step = np.diff(counter)
    index = np.arange(clock.size)
    grid_ms = (clock - np.polyval(np.polyfit(index, clock, 1), index)) * 1000.0
    return {'subject': row.subject, 'task': row.task, 'frames': len(host),
            'dropped': int((counter_step[counter_step > 0] - 1).sum()),
            'backwards': int((counter_step <= 0).sum()),
            'fps': 1000.0 / float(np.median(step_ms)),
            'step_min_ms': float(step_ms.min()), 'step_max_ms': float(step_ms.max()),
            'grid_max_ms': float(np.abs(grid_ms).max()), 'span_s': float(clock[-1]),
            'drift_ppm': b * 1e6, 'lag_med_ms': float(np.median(lag_ms)),
            'lag_max_ms': float(lag_ms.max()), 'lag_over_10ms': float((lag_ms > 10).mean())}


timing = pd.DataFrame([timing_audit(r) for r in recordings.itertuples()])

print(f'frames dropped by the camera counter: {timing.dropped.sum()} | counter steps going '
      f'backwards: {timing.backwards.sum()}')
print(f'frame rate on the camera clock {timing.fps.min():.3f}-{timing.fps.max():.3f} fps | '
      f'interval {timing.step_min_ms.min():.2f}-{timing.step_max_ms.max():.2f} ms | duration '
      f'{timing.span_s.min():.1f}-{timing.span_s.max():.1f} s')
print(f'against a uniform grid at each recording\'s own rate: worst '
      f'{timing.grid_max_ms.max():.3f} ms in any recording')
print(f'camera clock against the PC clock: {timing.drift_ppm.median():+.0f} ppm in the median '
      f'recording')
print(f'PC arrival lag above the envelope: median {timing.lag_med_ms.median():.2f} ms, worst '
      f'{timing.lag_max_ms.max():.1f} ms')
late = timing[timing.lag_max_ms > 20]
if len(late):
    print('\nrecordings whose PC stamps run more than 20 ms late at some point:')
    print(late[['subject', 'task', 'span_s', 'lag_med_ms', 'lag_max_ms',
                'lag_over_10ms']].round(3).to_string(index=False))
print()
print(timing[['subject', 'task', 'frames', 'fps', 'span_s', 'dropped', 'grid_max_ms',
              'lag_med_ms']].round(3).to_string(index=False))

frames dropped by the camera counter: 0 | counter steps going backwards: 0
frame rate on the camera clock 29.955-29.978 fps | interval 33.34-273.38 ms | duration 60.5-241.2 s
against a uniform grid at each recording's own rate: worst 128.877 ms in any recording
camera clock against the PC clock: +1 ppm in the median recording
PC arrival lag above the envelope: median 0.09 ms, worst 802.7 ms

recordings whose PC stamps run more than 20 ms late at some point:
subject   task  span_s  lag_med_ms  lag_max_ms  lag_over_10ms
     01 motion  65.597       0.112     266.927          0.004
     02  still 210.445       0.072     509.717          0.003
     03 motion  60.830      80.589     238.559          0.937
     03  still 241.158       0.085     246.086          0.001
     04 motion  61.558       0.095     665.997          0.011
     05  still 180.568       0.091     561.529          0.004
     06 motion  60.511       0.087      35.019          0.003
     06  still 193.809       0.088     802

## 4. The oximeter: its record, its clock, and the samples it lost
The reference is a finger pulse oximeter's plethysmographic waveform, stored with the PC's
time of each sample and a sample count. Three things about the record have to be established
before a beat is read from it.

**What the record holds.** The waveform arrives as a ragged array of records rather than one
vector, and flattening it yields more values than the stored count claims in most recordings,
so the count is not taken as the length. Two encodings exist across the MR-NIRP sets: one
stores the waveform with its top bit set, which has to be removed, and one stores it plainly.
Which one this is, is decided from the values themselves rather than from a README.

**Its clock.** The stamps record when the PC read each sample, not when the device took it.
The reader collects the serial stream in bursts and also loses samples, and both matter for
HRV: dating samples by their stamps adds the reading jitter to every beat, while dating them
by their index alone puts every sample after a loss one period early, with the error
accumulating over the recording. The device's own clock is rebuilt instead, because it is
regular and the losses are whole samples: against a 60 Hz grid the stamps form a floor of
prompt reads with late ones above it, and that floor steps up by exactly one period at each
loss. A loss is accepted only when every sample for the next half second arrives at least one
period late, so a single late read cannot pass for one.

**Whether the rebuilt clock is right.** The positive control answers it: losses are injected
into a complete stretch at a known rate and the rebuilt clock is asked to put the surviving
samples back where they came from. The share it misplaces is the measure, and it is reported
per recording because a reference whose clock is doubted cannot support an HRV claim.

The reference's span is compared with the frames' as well, since a reference that stops early
limits what a recording can be scored on.

In [5]:
def read_oximeter(folder):
    """The waveform and the PC's read time of each sample, from pulseOx.mat.

    The record is stored as a ragged object array here and as a plain integer array in the car
    dataset from the same group, so it is flattened rather than converted element by element.
    Its values carry no offset in this dataset, where the car's and Rice's carry the top bit
    set, so an offset is removed only where the data shows one.
    """
    mat = sio.loadmat(folder / 'PulseOX' / 'pulseOx.mat', squeeze_me=True)
    record = np.atleast_1d(mat['pulseOxRecord'])
    wave = np.concatenate([np.atleast_1d(np.asarray(v)).ravel() for v in record]).astype(float)
    stamps = np.atleast_1d(mat['pulseOxTime']).astype(np.float64)
    claimed = int(np.atleast_1d(mat['numPulseSample'])[0])
    offset = 128.0 if wave.min() >= 128 else 0.0
    return wave - offset, stamps, claimed, offset


def sample_slots(stamps, fs=OX_FS, lookahead=LOSS_LOOKAHEAD, tol=LOSS_TOL_S):
    """Places every read sample in its slot on the device's own grid (NB_P3_33 section 3).

    slots[k] is k plus the samples lost before sample k. A stamp minus its slot time sits on a
    floor of prompt reads, and the floor steps up one period at each lost sample. A loss is
    accepted only when all of the next `lookahead` samples sit at least a period (less `tol`)
    above the floor of the `lookahead` before them, so one late read never passes for a loss.
    """
    period = 1.0 / fs
    n = stamps.size
    slots = np.empty(n, dtype=np.int64)
    floor = np.empty(n)
    below = np.zeros(n, dtype=bool)
    lost = 0
    for k in range(n):
        hi = min(n, k + lookahead)
        ahead = stamps[k:hi] - (np.arange(k, hi) + lost) * period
        before = ahead.min() if k == 0 else floor[max(0, k - lookahead):k].min()
        while hi - k >= 5 and ahead.min() - before >= period - tol:
            lost += 1
            ahead = ahead - period
        below[k] = ahead.min() - before <= -(period - tol)
        slots[k] = k + lost
        floor[k] = stamps[k] - slots[k] * period
    steps_down = int(below[0]) + int((below[1:] & ~below[:-1]).sum())
    return slots, steps_down


def rebuild_clock(stamps, fs=OX_FS):
    """Slots, sample times on the PC clock, read delays and the device's measured rate."""
    slots, steps_down = sample_slots(stamps, fs)
    grid_s = slots / fs
    a, b = lower_line(grid_s, stamps - grid_s)
    times = a + (1.0 + b) * grid_s
    return {'slots': slots, 'times': times, 'delay': stamps - times,
            'rate_hz': fs / (1.0 + b), 'steps_down': steps_down}


def loss_control(stamps, slots, rate=CONTROL_LOSS_RATE, seed=0):
    """A positive control: drop samples at a known rate and see whether the rule finds them.

    The real losses are left where they are and further ones are injected on top, under this
    recording's own read delays, so the control measures the rule against this recording rather
    than against an ideal one.
    """
    rng = np.random.default_rng(seed)
    drop = rng.random(stamps.size) < rate
    kept = ~drop
    if kept.sum() < 100:
        return float('nan'), float('nan')
    found, _ = sample_slots(stamps[kept])
    truth = slots[kept] - slots[kept][0]
    found = found - found[0]
    misplaced = float(np.mean(found != truth))
    return float(drop.sum()), misplaced


def oximeter_audit(row, frame_t):
    """File checks, the rebuilt clock and coverage of one recording."""
    wave, stamps, claimed, offset = read_oximeter(row.folder)
    clock = rebuild_clock(stamps)
    gaps = np.diff(clock['slots']) - 1
    injected, misplaced = loss_control(stamps, clock['slots'])
    index = np.arange(stamps.size)
    straight = np.polyval(np.polyfit(index, stamps, 1), index)
    return {'subject': row.subject, 'task': row.task, 'samples': int(stamps.size),
            'n_values': int(wave.size), 'claimed': claimed, 'offset_removed': offset,
            'value_min': float(wave.min()), 'value_max': float(wave.max()),
            'rate_hz': clock['rate_hz'],
            'lost': int(gaps.sum()), 'loss_events': int((gaps > 0).sum()),
            'longest_gap': int(gaps.max()) if gaps.size else 0,
            'steps_down': clock['steps_down'],
            'delay_med_ms': float(np.median(clock['delay']) * 1000.0),
            'delay_p95_ms': float(np.percentile(clock['delay'], 95) * 1000.0),
            'stamp_wander_ms': float(np.ptp(stamps - straight) * 1000.0),
            'control_injected': injected, 'control_misplaced': misplaced,
            'starts_after_s': float(clock['times'][0] - frame_t[0]),
            'short_by_s': float(frame_t[-1] - clock['times'][-1])}


FRAME_TIMES, OXIMETER = {}, {}
rows = []
for r in recordings.itertuples():
    key = (r.subject, r.task)
    FRAME_TIMES[key], _ = clock_for_frames(r.folder, int(r.frames))
    wave, stamps, _, _ = read_oximeter(r.folder)
    clock = rebuild_clock(stamps)
    OXIMETER[key] = {'wave': wave, 'slots': clock['slots'], 'times': clock['times']}
    rows.append(oximeter_audit(r, FRAME_TIMES[key]))
oximeter = pd.DataFrame(rows)

print(f'the record holds {oximeter.n_values.min()}-{oximeter.n_values.max()} values against a '
      f'claimed {oximeter.claimed.min()}-{oximeter.claimed.max()}; flattening the ragged array '
      f'yields {int((oximeter.n_values != oximeter.claimed).sum())} recordings where the two '
      f'differ')
offsets_removed = sorted(oximeter.offset_removed.unique())
print(f'values run {oximeter.value_min.min():.0f}-{oximeter.value_max.max():.0f} after removing '
      f'an offset of {", ".join(f"{o:.0f}" for o in offsets_removed)}')
print(f'measured rate {oximeter.rate_hz.min():.3f}-{oximeter.rate_hz.max():.3f} Hz against a '
      f'nominal {OX_FS:g}')
print(f'lost samples {oximeter.lost.sum()} over {len(oximeter)} recordings | per recording '
      f'median {oximeter.lost.median():.0f}, worst {oximeter.lost.max()} | longest gap '
      f'{oximeter.longest_gap.max()} samples | steps down, impossible under the loss model: '
      f'{oximeter.steps_down.sum()}')
print(f'read delay above the rebuilt clock: median {oximeter.delay_med_ms.median():.1f} ms, '
      f'95th percentile {oximeter.delay_p95_ms.median():.1f} ms | the stamps wander '
      f'{oximeter.stamp_wander_ms.median():.0f} ms peak to peak around a straight line')
print(f'positive control, losses injected at {CONTROL_LOSS_RATE:.0%}: misplaced '
      f'{oximeter.control_misplaced.median():.2%} of samples in the median recording, '
      f'{oximeter.control_misplaced.max():.2%} at worst')
print(f'\nthe reference starts {oximeter.starts_after_s.min():.2f}-'
      f'{oximeter.starts_after_s.max():.2f} s after the first frame and ends '
      f'{oximeter.short_by_s.min():+.2f} to {oximeter.short_by_s.max():+.2f} s before the last')
print()
print(oximeter[['subject', 'task', 'samples', 'rate_hz', 'lost', 'longest_gap', 'delay_med_ms',
                'control_misplaced', 'starts_after_s', 'short_by_s']].round(3).to_string(index=False))

the record holds 3623-14440 values against a claimed 3580-14382; flattening the ragged array yields 12 recordings where the two differ
values run 0-112 after removing an offset of 0
measured rate 59.876-59.879 Hz against a nominal 60
lost samples 1081 over 15 recordings | per recording median 35, worst 383 | longest gap 47 samples | steps down, impossible under the loss model: 0
read delay above the rebuilt clock: median 0.0 ms, 95th percentile 1.3 ms | the stamps wander 554 ms peak to peak around a straight line
positive control, losses injected at 1%: misplaced 0.00% of samples in the median recording, 0.03% at worst

the reference starts -0.02-0.02 s after the first frame and ends -0.15 to +14.31 s before the last

subject   task  samples  rate_hz  lost  longest_gap  delay_med_ms  control_misplaced  starts_after_s  short_by_s
     01 motion     3910   59.877    19           19         0.035                0.0          -0.016       0.013
     01  still    10703   59.877    52        

## 5. Reference beats, and the HRV each recording supports
The waveform goes onto the rebuilt clock with lost samples interpolated and marked, and the
beats are read with the fiducial and the rules the project's HRV work already uses: the
steepest point of the upstroke, refined between samples by a three-point parabola, with a
minimum spacing of 55% of the recording's own dominant beat period read from its spectrum.

An interval is dropped when it lies outside 35-180 bpm, when either of its beats has an
interpolated or a clipped sample within two samples of its upstroke, or when it differs by
more than 25% from the median of the seven intervals centred on it. Successive differences are
taken only between neighbouring kept intervals, so a single artefact cannot manufacture a
large difference.

Whether a recording can carry an HRV claim at all is then decided by two tests, not by the
numbers coming out plausible: at least 90% of its intervals kept, and at least 90 s of them. A
thin record's gaps are not random - they sit where the waveform was lost or clipped, which can
follow the breathing or the hand - and 90 s is already short for SDNN. This is pulse-rate
variability at the finger rather than R-R variability from an ECG, which is the like-for-like
reference for a camera, since a camera also sees the pulse and not the R wave.

In [6]:
BEAT_BAND = butter(2, [0.5, 8.0], btype='band', fs=OX_FS, output='sos')
HR_BAND = butter(2, [HR_MIN_BPM / 60.0, HR_MAX_BPM / 60.0], btype='band', fs=OX_FS, output='sos')


def on_grid(wave, slots):
    """The waveform on the device's own grid, with lost samples interpolated and marked."""
    grid = np.full(int(slots[-1]) + 1, np.nan)
    grid[slots[:wave.size]] = wave[:slots.size]
    missing = np.isnan(grid)
    if missing.any():
        idx = np.arange(grid.size)
        grid[missing] = np.interp(idx[missing], idx[~missing], grid[~missing])
    return grid, missing


def dominant_period(signal, fs):
    """The dominant beat period in samples, from a 16 s Welch spectrum."""
    f, p = welch(signal - signal.mean(), fs=fs, nperseg=min(len(signal), int(16 * fs)))
    band = (f >= HR_MIN_BPM / 60.0) & (f <= HR_MAX_BPM / 60.0)
    return fs / f[band][np.argmax(p[band])]


def refine(y, i):
    """Sub-sample position of a maximum from a parabola through three points."""
    if i <= 0 or i >= len(y) - 1:
        return float(i)
    left, centre, right = float(y[i - 1]), float(y[i]), float(y[i + 1])
    denominator = left - 2.0 * centre + right
    if abs(denominator) < 1e-12:
        return float(i)
    off = 0.5 * (left - right) / denominator
    return float(i) + (off if abs(off) <= 0.5 else 0.0)


def steepest_rises(signal, fs, period):
    """Sub-sample positions of the steepest upstrokes: NB_P3_33's fiducial and rules."""
    slope = np.gradient(signal)
    idx, _ = find_peaks(slope, distance=max(1, int(BEAT_SPACING * period)))
    if idx.size < 3:
        return np.array([]), slope
    local = pd.Series(slope[idx]).rolling(9, center=True, min_periods=3).median().to_numpy()
    idx = idx[slope[idx] >= 0.4 * local]
    return np.array([refine(slope, int(i)) for i in idx]), slope


def find_beats(grid, missing, fs=OX_FS):
    """One row per beat: its steepest-upstroke time, and whether it borders a lost sample."""
    x = sosfiltfilt(BEAT_BAND, grid - grid.mean())
    period = dominant_period(sosfiltfilt(HR_BAND, grid - grid.mean()), fs)
    positions, _ = steepest_rises(x, fs, period)
    rows = []
    for p in positions:
        i = int(round(p))
        around = slice(max(0, i - GAP_GUARD), i + GAP_GUARD + 1)
        rows.append({'slope_s': p / fs, 'near_gap': bool(missing[around].any()),
                     'clipped': bool(((grid[around] <= grid.min()) |
                                      (grid[around] >= grid.max())).any())})
    return pd.DataFrame(rows, columns=['slope_s', 'near_gap', 'clipped']), period


def intervals(beats):
    """Beat-to-beat intervals with NB_P3_33's artefact rules applied."""
    t = beats.slope_s.to_numpy()
    ibi = np.diff(t)
    flagged = (beats.near_gap | beats.clipped).to_numpy()
    in_range = (ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)
    clean = ~flagged[:-1] & ~flagged[1:]
    candidate = in_range & clean
    local = (pd.Series(np.where(candidate, ibi, np.nan))
             .rolling(7, center=True, min_periods=3).median().to_numpy())
    consistent = np.abs(ibi - local) <= IBI_LOCAL_TOL * local
    return pd.DataFrame({'t': t[1:], 'ibi': ibi, 'kept': candidate & consistent})


def hrv_metrics(table):
    """Mean heart rate, SDNN and RMSSD from the kept intervals."""
    ibi, kept = table.ibi.to_numpy(), table.kept.to_numpy()
    pair = kept[1:] & kept[:-1]
    diffs, values = np.diff(ibi)[pair], ibi[kept]
    if values.size < 8 or diffs.size < 8:
        return dict.fromkeys(('hr_bpm', 'sdnn_ms', 'rmssd_ms'), np.nan)
    return {'hr_bpm': 60.0 / values.mean(), 'sdnn_ms': values.std(ddof=1) * 1000.0,
            'rmssd_ms': float(np.sqrt(np.mean(diffs ** 2)) * 1000.0)}


REF_BEATS, BEAT_TABLES = {}, {}
rows = []
for r in recordings.itertuples():
    key = (r.subject, r.task)
    ox = OXIMETER[key]
    grid, missing = on_grid(ox['wave'], ox['slots'])
    beats, period = find_beats(grid, missing)
    table = intervals(beats)
    BEAT_TABLES[key] = table
    REF_BEATS[key] = np.interp(beats.slope_s.to_numpy() * OX_FS, ox['slots'], ox['times'])
    kept_s = float(table.loc[table.kept, 'ibi'].sum())
    metrics = hrv_metrics(table)
    rows.append({'subject': r.subject, 'task': r.task, 'beats': len(beats),
                 'intervals': len(table), 'kept_share': float(table.kept.mean()),
                 'kept_s': kept_s, 'hr_spectral_bpm': 60.0 * OX_FS / period, **metrics,
                 'carries_hrv': bool(table.kept.mean() >= HRV_MIN_KEPT
                                     and kept_s >= HRV_MIN_SECONDS)})
beat_audit = pd.DataFrame(rows)

print(f'beats per recording {beat_audit.beats.min()}-{beat_audit.beats.max()} | intervals kept: '
      f'median {beat_audit.kept_share.median():.1%}, lowest {beat_audit.kept_share.min():.1%}')
print(f'heart rate {beat_audit.hr_bpm.min():.1f}-{beat_audit.hr_bpm.max():.1f} bpm | RMSSD '
      f'{beat_audit.rmssd_ms.min():.1f}-{beat_audit.rmssd_ms.max():.1f} ms | SDNN '
      f'{beat_audit.sdnn_ms.min():.1f}-{beat_audit.sdnn_ms.max():.1f} ms')
print(f'references carrying HRV (at least {HRV_MIN_KEPT:.0%} of intervals and '
      f'{HRV_MIN_SECONDS:.0f} s of them): {int(beat_audit.carries_hrv.sum())} of '
      f'{len(beat_audit)} | by task '
      f'{beat_audit.groupby("task").carries_hrv.sum().to_dict()}')
print()
print(beat_audit[['subject', 'task', 'beats', 'kept_share', 'kept_s', 'hr_bpm', 'sdnn_ms',
                  'rmssd_ms', 'carries_hrv']].round(2).to_string(index=False))

beats per recording 55-298 | intervals kept: median 96.5%, lowest 87.8%
heart rate 53.5-81.3 bpm | RMSSD 25.7-95.0 ms | SDNN 38.8-81.9 ms
references carrying HRV (at least 90% of intervals and 90 s of them): 7 of 15 | by task {'motion': 0, 'still': 7}

subject   task  beats  kept_share  kept_s  hr_bpm  sdnn_ms  rmssd_ms  carries_hrv
     01 motion     86        0.96   61.68   79.76    51.25     43.22        False
     01  still    241        0.99  175.88   81.19    39.24     38.97         True
     02 motion     70        0.96   55.69   71.10    81.87     84.01        False
     02  still    229        0.99  204.41   66.04    78.57     78.92         True
     03 motion     72        0.96   56.69   71.97    46.74     38.42        False
     03  still    298        0.96  229.92   74.37    58.07     43.16         True
     04 motion     68        0.97   57.83   67.44    78.03     60.00        False
     04  still    195        1.00  178.99   65.03    67.90     68.77         True
     05  

## 6. The frames through the app's own code
Before it reads anything, the app finds the face with MediaPipe's landmarker on 12 frames
spread over the capture, holds one box for the whole capture - the median landmark box, made
square and padded - and judges the capture on it: whether the padded box fits the frame, and
whether the face was found in the samples, alone, and inside the box. Those functions run here
unchanged, on frames demosaiced with the code section 2 found.

Every frame is then cropped with that held box to the app's 72x72 crop by the app's own
`crop_resize`, clamped to the frame where the padded box overflows it, which is what the app
would crop if its framing gate let the capture through. From the crops come the app's motion
verdict and its classical POS reading, each under the app's own gates.

The same pass also keeps a second colour trace, taken at the sensor's full depth from the same
region of the same frames. The app's contract is an 8-bit crop, which is what a browser hands
it, so the crops themselves stay 8-bit and the reading above is the app's; the full-depth trace
is only kept so that section 8 can ask what the 8-bit contract costs, with the two traces
differing in nothing else.

In [7]:
def demosaic(raw):
    return cv2.cvtColor(to_8bit(raw), BAYER_CODE)


def crops_of(row, box):
    """Every frame as the app's 8-bit crop, and the same crop's colour trace at full precision.

    The app's contract is an 8-bit crop, which is what a browser hands it, so the crops are made
    the way the app would make them: the high byte first, then the same crop and resize. The
    sensor writes 12 bits, though, and the four bits the high byte discards are not nothing when
    the pulse is a fraction of a percent of the mean - the docstring of the trace helper says
    8-bit quantisation sits an order of magnitude below that pulse, which is an assumption worth
    testing rather than inheriting. So the same crop is also taken at full depth, from one read
    of the frame, and its central region averaged by the same helper. Section 7 compares the two.
    """
    def one(i):
        raw = read_raw(frame_path(row.folder, i))
        return crop_resize(demosaic(raw), box), crop_resize(cv2.cvtColor(raw, BAYER_CODE), box)
    crops = np.empty((int(row.frames), TARGET_SIZE, TARGET_SIZE, 3), dtype=np.uint8)
    trace = np.empty((int(row.frames), 3), dtype=np.float64)
    with ThreadPoolExecutor(FRAME_WORKERS) as pool:
        for i, (crop8, crop16) in enumerate(pool.map(one, range(int(row.frames)))):
            crops[i] = crop8
            trace[i] = rgb_trace(crop16[None].astype(np.float64))[0]
    return crops, trace


def frame_audit(row):
    """The app's detector, held box, gates and POS reading on one recording."""
    samples = [demosaic(read_raw(frame_path(row.folder, int(i))))
               for i in detection_positions(int(row.frames))]
    box, found, faces = stable_box(samples, LANDMARKER)
    out = {'subject': row.subject, 'task': row.task, 'found': int(found), 'of': len(samples)}
    if box is None:
        return out, None, None, None
    framing, report, _ = framing_verdict(box, int(row.width), int(row.height))
    checks, _, summary = face_verdict(faces, box)
    subject = np.array([f[0] for f in faces if f])
    crops, trace = crops_of(row, box)
    reading = HRSpectral().estimate(crops, NOMINAL_FPS)
    motion, _, motion_summary = motion_verdict(crops, NOMINAL_FPS)
    out.update({'face_h': float(np.median(subject[:, 3] - subject[:, 1])),
                'side_lost': float(report['frac_side_lost']), 'framing': framing,
                'face_checks': checks, 'motion': motion,
                'motion_px': float((motion_summary or {}).get('amp_median', np.nan)),
                'pos_hr': float(reading.value), 'pos_status': reading.status,
                'pos_confidence': float(reading.confidence),
                'brightness': float(np.median([crops[i].mean() for i in
                                               np.linspace(0, len(crops) - 1, 9).astype(int)]))})
    return out, box, crops, trace


CROPS, BOXES, TRACES16 = {}, {}, {}
rows = []
t_start = time.time()
for k, r in enumerate(recordings.itertuples(), 1):
    key = (r.subject, r.task)
    print(f'\rframe pass {k}/{len(recordings)}: {r.subject}/{r.task}   ', end='', flush=True)
    audit, box, crops, trace = frame_audit(r)
    rows.append(audit)
    if box is not None:
        BOXES[key], CROPS[key], TRACES16[key] = box, crops, trace
frames_audit = pd.DataFrame(rows)
print(f'\rframe pass: {len(CROPS)} of {len(recordings)} recordings in '
      f'{(time.time() - t_start) / 60:.1f} min')

print(f'face found in all {N_DETECT_FRAMES} samples: '
      f'{int((frames_audit.found == N_DETECT_FRAMES).sum())}/{len(frames_audit)}')
print(f'the face spans {frames_audit.face_h.min():.0%}-{frames_audit.face_h.max():.0%} of the '
      f'frame\'s height; padded by {CROP_PADDING:g} the held box loses '
      f'{frames_audit.side_lost.min():.0%}-{frames_audit.side_lost.max():.0%} of its side')
print(f'app gates: framing {frames_audit.framing.value_counts().to_dict()} | face checks '
      f'{frames_audit.face_checks.value_counts().to_dict()} | motion '
      f'{frames_audit.motion.value_counts().to_dict()} (crop displacement '
      f'{frames_audit.motion_px.min():.2f}-{frames_audit.motion_px.max():.2f} px)')
print(f'POS reading: {frames_audit.pos_status.value_counts().to_dict()} | brightness '
      f'{frames_audit.brightness.min():.0f}-{frames_audit.brightness.max():.0f}')
print()
merged = frames_audit.merge(beat_audit[['subject', 'task', 'hr_bpm', 'carries_hrv']],
                            on=['subject', 'task'])
merged['pos_error'] = merged.pos_hr - merged.hr_bpm
print(merged[['subject', 'task', 'found', 'face_h', 'side_lost', 'framing', 'motion', 'motion_px',
              'brightness', 'pos_hr', 'hr_bpm', 'pos_error', 'pos_status']]
      .round(2).to_string(index=False))

frame pass: 15 of 15 recordings in 4.3 min
face found in all 12 samples: 15/15
the face spans 57%-79% of the frame's height; padded by 0.6 the held box loses 0%-21% of its side
app gates: framing {'REJECT': 13, 'WARN': 2} | face checks {'ACCEPT': 15} | motion {'ACCEPT': 15} (crop displacement 0.03-0.12 px)
POS reading: {'ok': 13, 'degraded_capture': 2} | brightness 46-120

subject   task  found  face_h  side_lost framing motion  motion_px  brightness  pos_hr  hr_bpm  pos_error       pos_status
     01 motion     12    0.77       0.20  REJECT ACCEPT       0.08       78.26   80.59   79.76       0.82               ok
     01  still     12    0.74       0.18  REJECT ACCEPT       0.03       73.35   81.48   81.19       0.29               ok
     02 motion     12    0.59       0.01    WARN ACCEPT       0.12       68.85   70.25   71.10      -0.85               ok
     02  still     12    0.57       0.00    WARN ACCEPT       0.04       68.61   65.40   66.04      -0.64               ok
     03 m

## 7. Camera beats against the reference
POS follows the light the skin reflects, and that light falls as blood volume rises, so the
systolic rise is where the POS pulse falls fastest. The camera's beat is therefore the steepest
rise of the inverted POS pulse - the same event as the finger's steepest upstroke - refined
between frames and read under section 5's spacing and slope rules.

The offset between the two, finger minus camera, holds the pulse's longer path to the finger,
the oximeter's own processing, and whatever part of each device's latency to the PC a lower
envelope cannot remove; the camera's stamp may also mark the start of its exposure rather than
the middle. It is found as the offset at which the median distance from each reference beat to
the nearest camera beat is smallest, and reported with the margin by which it beats the best
rival alignment, so that a slip of a whole beat shows as a small margin rather than hiding in
a count of matches.

What the comparison then measures is timing, not rate: the share of reference beats matched
within 150 ms, the median distance of those matches, and the camera's SDNN and RMSSD against
the reference's. The quadrature rule says what to expect from timing noise alone - independent
jitter of SD s adds 2s squared to SDNN squared and 6s squared to RMSSD squared - so RMSSD is
the measure that fails first, and by how much it exceeds the reference says how much jitter is
in the beats.

In [8]:
def camera_beats(key):
    """The camera's beat times on the PC clock, from the app's POS pulse inverted."""
    crops, fps = CROPS[key], NOMINAL_FPS
    pulse = -pos(rgb_trace(crops), fps)
    if not np.any(pulse):
        return np.array([]), pulse
    positions, _ = steepest_rises(pulse, fps, dominant_period(pulse, fps))
    return np.interp(positions, np.arange(FRAME_TIMES[key].size), FRAME_TIMES[key]), pulse


rows = []
for key in sorted(CROPS):
    cam, pulse = camera_beats(key)
    ref = REF_BEATS[key]
    offset, gap, matched, margin = align_offset(cam, ref)
    gaps = nearest_gaps(cam + offset, ref) if cam.size >= 3 else np.array([np.nan])
    close = gaps[gaps <= MATCH_TOLERANCE_S]
    table = BEAT_TABLES[key]
    cam_table = intervals(pd.DataFrame({'slope_s': cam + offset, 'near_gap': False,
                                        'clipped': False}))
    rows.append({'subject': key[0], 'task': key[1], 'camera_beats': int(cam.size),
                 'reference_beats': int(ref.size),
                 'per_reference': float(cam.size / max(ref.size, 1)),
                 'offset_ms': offset * 1000.0, 'gap_ms': gap * 1000.0, 'matched': matched,
                 'margin_ms': margin,
                 'timing_ms': float(np.median(close) * 1000.0) if close.size else np.nan,
                 'camera_rmssd_ms': hrv_metrics(cam_table)['rmssd_ms'],
                 'reference_rmssd_ms': hrv_metrics(table)['rmssd_ms'],
                 'camera_sdnn_ms': hrv_metrics(cam_table)['sdnn_ms'],
                 'reference_sdnn_ms': hrv_metrics(table)['sdnn_ms']})
alignment = pd.DataFrame(rows)
alignment['sdnn_error'] = alignment.camera_sdnn_ms / alignment.reference_sdnn_ms - 1
alignment['rmssd_error'] = alignment.camera_rmssd_ms / alignment.reference_rmssd_ms - 1

print(f'camera beats per reference beat {alignment.per_reference.median():.2f} | matched within '
      f'{MATCH_TOLERANCE_S * 1000:.0f} ms {alignment.matched.median():.0%} in the median '
      f'recording | timing {alignment.timing_ms.median():.1f} ms '
      f'({alignment.timing_ms.quantile(0.25):.1f}-{alignment.timing_ms.quantile(0.75):.1f})')
print(f'offsets {alignment.offset_ms.min():+.0f} to {alignment.offset_ms.max():+.0f} ms | the '
      f'best rival alignment is {alignment.margin_ms.min():.0f}-{alignment.margin_ms.max():.0f} '
      f'ms worse')
print(f'SDNN error {alignment.sdnn_error.median():+.1%} in the median recording, RMSSD '
      f'{alignment.rmssd_error.median():+.1%}')
print()
print(alignment[['subject', 'task', 'per_reference', 'offset_ms', 'gap_ms', 'matched',
                 'timing_ms', 'reference_sdnn_ms', 'sdnn_error', 'reference_rmssd_ms',
                 'rmssd_error']].round(3).to_string(index=False))

camera beats per reference beat 1.00 | matched within 150 ms 68% in the median recording | timing 41.6 ms (19.5-46.6)
offsets -75 to +235 ms | the best rival alignment is 58-241 ms worse
SDNN error +20.5% in the median recording, RMSSD +66.6%

subject   task  per_reference  offset_ms  gap_ms  matched  timing_ms  reference_sdnn_ms  sdnn_error  reference_rmssd_ms  rmssd_error
     01 motion          0.988      -25.0  96.691    0.512     41.570             51.249       0.302              43.215        1.231
     01  still          1.004       10.0  56.061    0.734     29.943             39.242       0.250              38.972        0.621
     02 motion          1.000       12.5  24.006    0.957     23.450             81.868       0.107              84.013        0.268
     02  still          1.000      -17.5  34.507    0.594     16.824             78.569       0.059              78.925        0.194
     03 motion          1.014       62.5  53.341    0.819     44.071             46.742    

## 8. What limits the timing
Heart rate comes out of these recordings within a couple of beats per minute, and the beats
are still timed several times less closely than the comparable public dataset's. Two
properties of this recorder could account for it, and this section takes them one at a time,
changing nothing else: the same frames, the same fiducial, the same rules, the same alignment.

**The frame clock.** POS reads a trace sample by sample with no notion of when each frame was
exposed, so an uneven frame spacing reaches its output as a time warp, placing a beat at the
position of a frame rather than at the moment it was taken. Section 3 measures that unevenness
directly, and it is orders of magnitude larger here than in the comparable dataset.
Interpolating the trace onto the recording's own mean rate before the beats are found removes
the warp, and only the warp.

**The depth of the crop.** The sensor writes 12 bits and the app's contract is an 8-bit crop,
so the high byte discards four bits of what the sensor measured - and a pulse is a fraction of
a percent of the mean brightness. The helper that reads the colour trace states as a design
assumption that 8-bit quantisation sits well over an order of magnitude below that pulse, which
is worth measuring rather than inheriting. Both traces come from the same region of the same
frames, so subtracting one from the other measures what the 8-bit contract actually costs, in
levels, against the pulse's own swing in the same units.

A reference-free quality number is reported beside both: the share of the pulse's power that
sits within 0.1 Hz of its own rate and second harmonic. A beat can only be timed as sharply as
the waveform it is read from, so if the recordings timed worst are the ones whose pulse is
least clean, the limit is the signal rather than the pipeline.

In [9]:
def beats_from(trace, times, even_grid):
    """Beat times from one colour trace, optionally resampled onto an even time grid.

    Two properties of this recorder could blunt a beat's position, and this takes them one at a
    time. The first is the frame clock: POS reads a trace sample by sample with no notion of when
    each frame was exposed, so an uneven spacing reaches its output as a time warp, and the
    camera clock here wanders up to a tenth of a second from a uniform grid where the comparable
    public dataset's never moved more than 0.05 ms. Interpolating the trace onto the recording's
    own mean rate before the beats are found removes that warp and nothing else. The second is
    depth, which the caller chooses by handing over the 8-bit or the full-depth trace. Everything
    else is held at the app's nominal rate, so the variants differ only in the factor under test.
    """
    if even_grid:
        rate = (times.size - 1) / (times[-1] - times[0])
        grid = times[0] + np.arange(times.size) / rate
        trace = np.column_stack([np.interp(grid, times, trace[:, channel])
                                 for channel in range(trace.shape[1])])
        times = grid
    pulse = -pos(trace, NOMINAL_FPS)
    if not np.any(pulse):
        return np.array([]), pulse
    positions, _ = steepest_rises(pulse, NOMINAL_FPS, dominant_period(pulse, NOMINAL_FPS))
    return np.interp(positions, np.arange(times.size), times), pulse


def against_reference(key, beats):
    """Timing error, match share and HRV error of one set of camera beats, as in cell 16."""
    ref = REF_BEATS[key]
    offset, _, matched, _ = align_offset(beats, ref)
    gaps = nearest_gaps(beats + offset, ref) if beats.size >= 3 else np.array([np.nan])
    close = gaps[gaps <= MATCH_TOLERANCE_S]
    camera = hrv_metrics(intervals(pd.DataFrame({'slope_s': beats + offset, 'near_gap': False,
                                                 'clipped': False})))
    reference = hrv_metrics(BEAT_TABLES[key])
    return {'timing_ms': float(np.median(close) * 1000.0) if close.size else np.nan,
            'matched': matched,
            'sdnn_error': camera['sdnn_ms'] / reference['sdnn_ms'] - 1,
            'rmssd_error': camera['rmssd_ms'] / reference['rmssd_ms'] - 1}


def band_snr_db(pulse, fs=NOMINAL_FPS):
    """Power within 0.1 Hz of the pulse rate and its second harmonic against the rest of the band.

    A beat's position can only be as sharp as the waveform it is read from, so this says how much
    of the pulse is pulse. It is a property of the signal alone and needs no reference.
    """
    f, power = welch(pulse - pulse.mean(), fs=fs, nperseg=min(len(pulse), int(16 * fs)))
    rate = fs / dominant_period(pulse, fs)
    band = (f >= HR_MIN_BPM / 60.0) & (f <= 4.0)
    peak = band & ((np.abs(f - rate) <= 0.1) | (np.abs(f - 2.0 * rate) <= 0.1))
    rest = band & ~peak
    return float(10.0 * np.log10(power[peak].sum() / power[rest].sum()))


def spread(x):
    """The 5th-to-95th percentile swing, which a single artefact cannot stretch."""
    low, high = np.percentile(x, [5, 95])
    return float(high - low)


VARIANTS = (('8-bit', False, False), ('8-bit, even grid', False, True),
            ('12-bit', True, False), ('12-bit, even grid', True, True))

rows = []
for key in sorted(CROPS):
    traces = {False: rgb_trace(CROPS[key]), True: TRACES16[key]}
    # Both traces are the same region of the same frames, so subtracting one from the other
    # measures what the app's 8-bit crop actually costs, instead of arguing it from pixel depth.
    quantisation = float(np.std(traces[False][:, 1] - traces[True][:, 1] / 256.0))
    pulse_levels = spread(bandpass(traces[True][:, 1], NOMINAL_FPS)) / 256.0
    for label, full_depth, even_grid in VARIANTS:
        beats, pulse = beats_from(traces[full_depth], FRAME_TIMES[key], even_grid)
        rows.append({'subject': key[0], 'task': key[1], 'variant': label,
                     'snr_db': band_snr_db(pulse), 'pulse_levels': pulse_levels,
                     'quantisation_levels': quantisation,
                     **against_reference(key, beats)})
limits = pd.DataFrame(rows)
by_variant = limits.groupby('variant', sort=False).median(numeric_only=True)
base = limits[limits.variant == '8-bit']

print(f'the pulse swings {limits.pulse_levels.min():.2f}-{limits.pulse_levels.max():.2f} 8-bit '
      f'levels in the green channel of the crop mean, and taking the high byte adds '
      f'{limits.quantisation_levels.min():.3f}-{limits.quantisation_levels.max():.3f} levels of '
      f'quantisation noise to that mean')
print()
print('median over the recordings, by what the beats were read from:')
print(by_variant[['timing_ms', 'matched', 'snr_db', 'sdnn_error',
                  'rmssd_error']].round(3).to_string())
print()
for label, _, _ in VARIANTS[1:]:
    other = limits[limits.variant == label].reset_index(drop=True)
    closer = int((other.timing_ms.to_numpy() < base.timing_ms.to_numpy()).sum())
    print(f'{label:>18}: timing {other.timing_ms.median():.1f} ms against '
          f'{base.timing_ms.median():.1f} ms, closer in {closer} of {len(base)} recordings')
print()
print(f'across recordings, a recording with a cleaner pulse is timed better: the timing '
      f'error against the band SNR is rho '
      f'{base.timing_ms.corr(base.snr_db, method="spearman"):+.2f}, and a recording timed '
      f'worse also overstates SDNN by more, rho '
      f'{base.timing_ms.corr(base.sdnn_error, method="spearman"):+.2f}')
print()
print(limits.pivot_table(index=['subject', 'task'], columns='variant', values='timing_ms',
                         sort=False).reindex(columns=[v[0] for v in VARIANTS]
                                            ).round(1).to_string())

the pulse swings 0.17-1.85 8-bit levels in the green channel of the crop mean, and taking the high byte adds 0.011-0.012 levels of quantisation noise to that mean

median over the recordings, by what the beats were read from:
                   timing_ms  matched  snr_db  sdnn_error  rmssd_error
variant                                                               
8-bit                 41.570    0.676   2.054       0.205        0.666
8-bit, even grid      37.215    0.676   2.053       0.205        0.666
12-bit                27.498    0.681   2.181       0.096        0.452
12-bit, even grid     27.499    0.681   2.180       0.096        0.447

  8-bit, even grid: timing 37.2 ms against 41.6 ms, closer in 4 of 15 recordings
            12-bit: timing 27.5 ms against 41.6 ms, closer in 13 of 15 recordings
 12-bit, even grid: timing 27.5 ms against 41.6 ms, closer in 13 of 15 recordings

across recordings, a recording with a cleaner pulse is timed better: the timing error against the ban

## 9. Findings and decision

**Layout (confirmed).** 15 recordings over 8 subjects - 8 still, 7 moving, one subject with
no moving recording - and 59,269 frames, 1,815-7,224 per recording, 60-241 s at a nominal
30 fps. Every frame of a recording is the same size, 640x640 in 13 recordings and 848x848 in
the two of one subject, which is why the size is read from each file's header. The colour
camera is the one whose time log has exactly one line per surviving frame, in 10 of 15
recordings against 1 for the other; it reports RAW12, 30 fps, gain 0 and a 30 ms shutter, and
the other camera reports MONO12, the near-infrared one whose frames are not in this copy. The
colour camera's log runs one line longer than its frames in 5 recordings, so logs are trimmed
to the saved frames.

**The frame.** The container is 16 bits wide and the sensor fills it to 65520 of 65535, a
12-bit value left-shifted by four. The layout is RGGB in all 15 recordings, read from the
data: the green pair's levels differ by 0.05 levels in the median recording against 19.09 for
the other pair, and OpenCV's code for it is `COLOR_BayerBG2RGB`. Byte order is not a
formality - handed the frame as the file stores it, the demosaic swapped the halves of every
pixel and raised the mean level from 21087 to 27953 while still returning something that looks
like a face - so the reader returns a native-order copy. The frames are dark: the median 8-bit
level is 31-112 across recordings, and true saturation is slight, 0.556% of pixels at the
sensor's ceiling at worst, where a count of 8-bit pixels at 255 would have claimed 11.4%.

**Frames are dated by the camera clock.** The counter shows no dropped frame in any recording.
The camera clock runs at 29.955 fps in 13 recordings and 29.978 in the two 848x848 ones, and a
uniform grid at each recording's own rate is within 0.045 ms of it in 14 of 15 - so a store may
keep frames as a uniform sequence at the measured rate. The exception is one moving recording
whose frames sit up to 128.9 ms from a uniform grid, with its PC stamps 80.6 ms late in the
median frame and more than 10 ms late in 94% of them; every other recording's stamps are prompt
(0.09 ms above the envelope in the median) with isolated stalls up to 0.8 s.

**The reference.** The waveform is a ragged array whose flattened length exceeds the stored
count in 12 of 15 recordings, so the count is not its length; the values run 0-112 with no
offset to remove, unlike the other MR-NIRP set. The rebuilt clock measures 59.876-59.879 Hz,
and the positive control puts injected losses back in place to 0.00% of samples in the median
recording and 0.03% at worst, so the clock is sound. 1,081 samples are lost over the 15
recordings, a median of 35 and a worst of 383 in one still recording - the same recording whose
reference also stops 14.3 s before its last frame.

**What the reference can support.** Heart rate 53.5-81.3 bpm, SDNN 38.8-81.9 ms, RMSSD
25.7-95.0 ms, with 96.5% of intervals kept in the median recording. Seven of the 15 carry HRV
under the 90%-and-90 s rule, and all seven are still recordings: the moving recordings are
about 60 s long and keep 55-62 s of intervals, so they fail on length alone whatever their
quality. The still recordings keep 154-230 s, which is longer than the project's other
HRV-capable reference set.

**The app's own code on these frames.** The landmarker finds a face in all 12 samples of all
15 recordings, never sees a second face, and the face stays inside the held box everywhere.
The faces are large - the landmark box spans 57-79% of the frame's height - so the padded box
overflows the frame by up to 21% of its side and the framing gate refuses 13 of 15, warning on
the other two. The motion gate accepts all 15, including every moving recording: the crop moves
0.03-0.06 px in the still recordings and 0.05-0.12 px in the moving ones, against a threshold
of 1.7 px. Whatever the moving task asked of the subjects, it is not motion this gate has to
refuse. POS reads 13 of 15 as ok and two as degraded, both from the subject whose frames carry
a bright clipped region, and its heart rate lands within -1.62 to +1.70 bpm of the reference
beat rate in every recording.

**Beat timing is the weak point.** The camera finds one beat per reference beat (ratio 1.00),
and after alignment 68% of reference beats have a camera beat within 150 ms in the median
recording, timed 41.6 ms from it (19.5-46.6 ms between the quartiles). That median is one
recording's own figure, and its alignment is tied across seven offsets over which the median
would read 30-42 ms (NB_P3_36 section 5). Offsets run -75 to +235 ms and beat their best rival
alignment by 58-241 ms, so no recording is aligned a whole beat out. SDNN comes out 20.5% above
the reference in the median recording and RMSSD 66.6% above - the asymmetry independent timing
jitter predicts, and about 23 ms of jitter by either route.

**The frame clock is not what costs the timing; the 8-bit contract costs about a quarter of
it.** Putting the trace on an even grid improves only 4 of 15 recordings, even though it
removes a grid deviation of up to 128.9 ms: the unevenness is slow drift, not per-beat jitter,
and a beat's neighbours are displaced with it. Reading the same region of the same frames at
the sensor's full depth times the beats better in 13 of 15 recordings, by a median 6.6 ms - a
median 24% of each recording's 8-bit error - with SDNN's excess falling from 20.5% to 9.6% and
RMSSD's from 66.6% to 45.2%. The section's two medians, 41.6 against 27.5 ms, would say a
third, but both are the same recording's figures and that recording's alignment is tied, so
their difference is not kept as a result; the per-recording comparison holds at 13 of 15
however the tie breaks (NB_P3_36 section 5). The quantisation this adds to the crop's colour
trace measures 0.011-0.012 levels against a pulse that swings 0.17-1.85 levels, so the trace
helper's design assumption holds as stated - the noise is far below the pulse - and the beats
are blunted anyway, because the fiducial is a slope, and differencing weights the broadband
noise the band-limited comparison does not see. The band SNR barely moves (2.05 to 2.18 dB),
and across recordings it tracks the timing error only weakly (rho -0.21), while the recordings
timed worst are the ones whose SDNN is most overstated (rho +0.44).

### Decision
**The dataset is adopted for evaluation, not for training, and the colour frames are stored as
the app's 8-bit crops.** Eight subjects from one site and one session cannot add training
signal to a corpus of several hundred, and the set carries no skin-tone or demographic labels
to balance against. What it does add is the project's longest HRV-capable references outside
the two-minute set: seven still recordings keeping 165-230 s of intervals, from a second
camera, a second site and a markedly darker exposure, which is the second independent read the
SDNN-at-rest question needs. The store should keep the 8-bit crops the app's contract defines,
even though the 12-bit source would time the beats better in 13 of 15 recordings, by a median
6.6 ms, because a browser capture never delivers more than 8 bits and a store that flattered
the pipeline would mis-state what the app can do; that gain is recorded here as the dataset's
own headroom, not as a pipeline change.

**What it does not add.** It is not a motion benchmark: the moving recordings move the crop a
twentieth of the gate's threshold and are too short to carry HRV, so the exercise set keeps
that role. It is not capture-realistic either - the framing gate refuses 13 of 15 because the
faces are too large for the padded box - so readings from it are what the app would report if
its gate let them through, and the gate's own behaviour has to be validated elsewhere.

**Carried into preprocessing.** Frames dated by the camera clock mapped onto the PC's through
the lower envelope, at each recording's measured rate; logs trimmed to the saved frames; frame
size from each file's header; the native-order copy before any demosaic; `COLOR_BayerBG2RGB`;
the high byte for the 8-bit image. Three recordings need handling rather than trust: the one
whose reference stops 14.3 s early, where the uncovered frames are marked rather than
trimmed, as every Phase-3 store marks them; the one whose PC stamps run 80 ms late, which the
camera clock already absorbs but which is flagged in the store; and the subject whose frames
carry a bright clipped region, two degraded readings and the worst beat timing of the set,
which is kept but excluded from the HRV set it already fails to qualify for. The
near-infrared frames are absent from this copy, so nothing here speaks to the near-infrared
half of the published dataset.

**The other MR-NIRP set stays parked.** The driving set was examined earlier and set aside: its
value would be motion refusal, which the exercise set already covers, and its single-wavelength
near-infrared frames fall outside the app's three-channel contract.